<a href="https://colab.research.google.com/github/mahigarg0403-blip/Customer-purchase-prediction_mahi_bhumika/blob/main/notebooks/%20step2_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#cell 1 : mount your drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
#cell 2 : download the dataset directly from UCI.
import urllib.request
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"
urllib.request.urlretrieve(url, "online_shoppers_intention.csv")

('online_shoppers_intention.csv', <http.client.HTTPMessage at 0x7e0b58b7c8c0>)

In [ ]:
#cell 3 : importing libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns',None)
pd.set_option('display.width',1000)
print("Import successful")

Import successful


In [ ]:
#cell 4 : load data
df = pd.read_csv("online_shoppers_intention.csv")
df.shape

(12330, 18)

In [ ]:
#cell 5 : dropping duplicates
df.drop_duplicates(inplace = True)
df.reset_index(drop = True)
print(f"Duplicate rows check : {df.duplicated().sum()}")
print(f"shape : {df.shape}")

Duplicate rows check : 0
shape : (12205, 18)


In [ ]:
#cell 6 : checking unique values for categorical columns
print(f"visitor_type : {df['VisitorType'].unique()}")
print(f"months : {df['Month'].unique()}")

visitor_type : ['Returning_Visitor' 'New_Visitor' 'Other']
months : ['Feb' 'Mar' 'May' 'Oct' 'June' 'Jul' 'Aug' 'Nov' 'Sep' 'Dec']


In [ ]:
#cell 7 : converting bool columns to numeric values
df['Revenue'] = df['Revenue'].astype(int)
df['Weekend'] = df['Weekend'].astype(int)

In [ ]:
#cell 8 : one-hot encoding for Logistic Regression
df_one_hot_encoded = pd.get_dummies(df, columns = ['Month','VisitorType'],drop_first = True)
df_one_hot_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12205 entries, 0 to 12329
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Administrative                 12205 non-null  int64  
 1   Administrative_Duration        12205 non-null  float64
 2   Informational                  12205 non-null  int64  
 3   Informational_Duration         12205 non-null  float64
 4   ProductRelated                 12205 non-null  int64  
 5   ProductRelated_Duration        12205 non-null  float64
 6   BounceRates                    12205 non-null  float64
 7   ExitRates                      12205 non-null  float64
 8   PageValues                     12205 non-null  float64
 9   SpecialDay                     12205 non-null  float64
 10  OperatingSystems               12205 non-null  int64  
 11  Browser                        12205 non-null  int64  
 12  Region                         12205 non-null  int6

In [ ]:
#cell 9 : checking unique values
df['Month'].unique()

array(['Feb', 'Mar', 'May', 'Oct', 'June', 'Jul', 'Aug', 'Nov', 'Sep',
       'Dec'], dtype=object)

In [ ]:
#cell 10 : label encoding for XGBoost & Random Forest
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'June':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
df['Month'] = df['Month'].map(month_map)

le = LabelEncoder()
df['VisitorType'] = le.fit_transform(df['VisitorType'])
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,1,1,1,1,2,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,2,1,2,2,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,4,1,9,3,2,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,2,3,2,2,4,2,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,2,3,3,1,4,2,1,0


In [ ]:
#cell 11 : Saving cleaned data
df_label_encoded = df.copy()
df_label_encoded.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/cleaned_data.csv', index = False)
print("Saved")

Saved


In [ ]:
#cell 12 : train-test-split for indices
train_index , test_index = train_test_split(
    df.index, test_size = 0.2, random_state = 42, stratify = df['Revenue']
)

In [ ]:
#cell 13 : train_test_splitting the datasets

#LR dataset
x_train_lr = df_one_hot_encoded.loc[train_index].drop('Revenue', axis = 1)
x_test_lr = df_one_hot_encoded.loc[test_index].drop('Revenue', axis = 1)
y_train_lr = df_one_hot_encoded.loc[train_index,'Revenue']
y_test_lr = df_one_hot_encoded.loc[test_index,'Revenue']

#Trees dataset (RF & XGB)
x_train_tree = df_label_encoded.loc[train_index].drop('Revenue', axis = 1)
x_test_tree = df_label_encoded.loc[test_index].drop('Revenue', axis = 1)
y_train_tree = df_label_encoded.loc[train_index,'Revenue']
y_test_tree = df_label_encoded.loc[test_index,'Revenue']

In [ ]:
#cell 14 : SMOTE on Training data for fixing class imbalance
smote = SMOTE(random_state = 42)
x_train_lr_sm, y_train_lr_sm = smote.fit_resample(x_train_lr, y_train_lr)
x_train_tree_sm, y_train_tree_sm = smote.fit_resample(x_train_tree, y_train_tree)

In [ ]:
#cell 15 : saving all sub-datasets
x_train_lr_sm.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/x_train_lr.csv', index = False)
x_test_lr.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/x_test_lr.csv', index = False)
y_train_lr_sm.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/y_train_lr.csv', index = False)
y_test_lr.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/y_test_lr.csv', index = False)
x_train_tree_sm.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/x_train_tree.csv', index = False)
x_test_tree.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/x_test_tree.csv', index = False)
y_train_tree_sm.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/y_train_tree.csv', index = False)
y_test_tree.to_csv('/content/drive/MyDrive/CPP-ML PROJECT-BHUMIKA-MAHI/y_test_tree.csv', index = False)
print("all saved")

all saved
